# Bonus Lab: CSV vs Parquet, PyArrow, and Compression

**Module:** Week 3 Day 1 (optional, outside class or stretch lane)  
**Estimated time:** 60 minutes  
**Format:** Individual  
**Kernel:** the shared repo-root `.venv`  
**AI-Free Zone:** write the code yourself.


> This is an optional bonus walkthrough. You are not expected to master this tool today.
> The goal is awareness: understand what problem this tool tries to solve, when it may help, and what trade-offs it introduces.


Tomorrow you move to BigQuery, which stores data in a columnar format and charges by bytes processed. This lab builds that intuition on your own machine, in four steps: scale a small dataset up, compare CSV and parquet, meet PyArrow, then run a compression showdown across codecs, the same experiment style used for comparing storage trade-offs in production teams.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

DATA_PATH = Path("data/taxi_trip_review.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Make sure you are running from the repo root."
    )

print("Data file found:", DATA_PATH)


In [ ]:
# Provided: a compact version of the Activity 2 cleaning policy.
raw = pd.read_csv(DATA_PATH)
clean = raw.copy()
clean.columns = [column.strip().lower() for column in clean.columns]
clean["pickup_ts"] = pd.to_datetime(clean["pickup_ts"], errors="coerce", utc=True)
for column in ["fare_amount", "tip_amount", "trip_distance"]:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")
clean["tip_amount"] = clean["tip_amount"].fillna(0.0).clip(lower=0.0)
clean = clean.dropna(subset=["trip_id", "pickup_borough", "fare_amount", "trip_distance", "pickup_ts"])
clean = clean[(clean["fare_amount"] > 0) & (clean["trip_distance"] > 0)]
clean = clean.drop_duplicates(subset=["trip_id"], keep="first").reset_index(drop=True)

print("clean rows:", len(clean))

## Part 1: Scale the Data Up

11 rows cannot show performance differences. Replicate the clean rows, then add small random jitter to the numeric columns so the values are not identical copies.

The default row count is intentionally moderate for beginner laptops. Increase `REPLICATIONS` to `10_000` if your laptop handles it comfortably.


In [ ]:
REPLICATIONS = 5_000  # increase to 10_000 if your laptop handles it comfortably
rng = np.random.default_rng(42)

# TODO: build big_df by concatenating `clean` REPLICATIONS times (pd.concat, ignore_index=True).
# TODO: add uniform jitter between -1.0 and 1.0 to fare_amount, and between -0.5 and 0.5 to trip_distance.
# TODO: keep fare_amount and trip_distance positive with .clip(lower=0.01).
# TODO: print the shape and expected row count:
# print("shape:", big_df.shape)
print("Expected rows:", len(clean) * REPLICATIONS)
raise NotImplementedError


## Part 2: Write CSV and Parquet, Compare Sizes

In [ ]:
csv_path = OUT_DIR / "big_trips.csv"
parquet_path = OUT_DIR / "big_trips.parquet"

# TODO: write big_df to csv_path with to_csv(index=False).
# TODO: write big_df to parquet_path with to_parquet(index=False).
# TODO: print each file size in MB using Path.stat().st_size.
raise NotImplementedError

## Part 3: Time Full Reads and Column-Pruned Reads

Use `time.perf_counter()` around each read. Run each timing cell twice and keep the second number, because the first read also pays a file cache penalty.

The two-column read is the columnar payoff: parquet can read selected columns without touching the rest of the file, while CSV must scan everything no matter how little you ask for. BigQuery charges by bytes processed for exactly this reason.

In [ ]:
def timed(label, fn):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f}s")
    return result

COLUMNS = ["pickup_borough", "fare_amount"]

# TODO: time pd.read_csv(csv_path) and pd.read_parquet(parquet_path).
# TODO: time pl.read_csv(csv_path) and pl.read_parquet(parquet_path).
# TODO: time the two-column reads:
#   pd.read_csv(csv_path, usecols=COLUMNS)
#   pd.read_parquet(parquet_path, columns=COLUMNS)
#   pl.scan_parquet(parquet_path).select(COLUMNS).collect()
raise NotImplementedError

## Part 4: Meet PyArrow, the Engine Underneath

When you called `to_parquet` and `read_parquet`, pandas handed the parquet work to a parquet engine such as **PyArrow**.

Arrow is a columnar memory format used across many modern data tools. PyArrow is the Python library that lets us work with Arrow tables and Parquet files directly.

For this course, remember:
- Arrow is mostly about how tabular data can live efficiently in memory.
- Parquet is mostly about how tabular data can live efficiently on disk.
- PyArrow is one common bridge between Python tools and those formats.

You can use PyArrow directly. Watch the types below.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(big_df)
print(type(table))
print("rows:", table.num_rows, "| columns:", table.num_columns)
print("Arrow in-memory size (MB):", round(table.nbytes / 1_000_000, 1))
table.schema

In [ ]:
# The same round trip pandas does for you, made explicit:
pq.write_table(table, OUT_DIR / "big_trips_pyarrow.parquet")
table_back = pq.read_table(OUT_DIR / "big_trips_pyarrow.parquet", columns=["pickup_borough", "fare_amount"])
print(type(table_back))
print("columns read:", table_back.column_names)
df_back = table_back.to_pandas()
df_back.head()

## Part 5: Compression Showdown

Parquet compresses each column chunk with a codec you choose. The engineering question is a three-way trade: **file size** vs **write time** vs **read time**.

| Codec | Reputation |
|---|---|
| snappy | The long-time default: fast, moderate compression. |
| zstd | The modern favorite: near-snappy speed, much smaller files. |
| gzip | Small files, but slower to write and read. |
| lz4 | Very fast, lighter compression. |
| none | No compression: fastest CPU-wise, biggest files. |

Run the experiment instead of trusting the reputation table. Loop over the codecs, write the same DataFrame with each one, and record size, write time, and read time.

In [ ]:
CODECS = ["snappy", "zstd", "gzip", "lz4", "none"]

results = []
# TODO: for each codec, write big_df with big_df.to_parquet(path, compression=codec)
#       (use compression=None when the codec string is "none").
# TODO: record the write seconds, the file size in MB, and the read seconds
#       (pd.read_parquet) in the results list as a dict per codec.
# TODO: build a DataFrame from results and sort it by size_mb.
raise NotImplementedError

Read your table like an engineer:

- Which codec would you pick for a **data lake that is written once and read thousands of times**?
- Which one for a **temporary staging file inside a pipeline** that is written and read exactly once?
- Storage costs money and so does compute. Which column of your table maps to which bill?

There is no single right answer, and that is the point: zstd is a strong modern default, snappy is everywhere in legacy lakes, and "none" can win inside a fast local pipeline. What matters is that you can defend the choice with the numbers you just produced.

## Reflection

Record these in your Day 1 README:

1. How much smaller was parquet than CSV for the same rows, and which codec won on size?
2. Did column pruning change which read was fastest? How does that connect to `SELECT *` in BigQuery tomorrow?
3. In one sentence: what is Arrow, and what is parquet?
4. Which compression codec would you set as your team default, and what evidence backs it?

## Cleanup

The generated files add up. Remove them when done:

```bash
rm -f outputs/big_trips*.csv outputs/big_trips*.parquet
```

## Bonus Tool Takeaways

| Topic | What students should remember | Do they need it today? |
|---|---|---|
| Dask | Pandas-like API, lazy execution, partitions, `.compute()` | No, awareness only |
| Modin | Tries to keep Pandas code while changing the execution engine | No, awareness only |
| FireDucks | Pandas-like acceleration, but environment-sensitive | No, awareness only |
| Parquet | Columnar storage, smaller files, column pruning, compression trade-offs | Yes, important for data engineering |
